# Hip implant loosening classifier — Colab runner

Trains a CNN to separate **Loose** from **Control** hip-implant X-rays
(Kaggle: `tawsifurrahman/aseptic-loose-hip-implant-xray-database`, 206 images).

Runtime → Change runtime type → **T4 GPU** before running anything.

In [ ]:
!nvidia-smi
import torch; print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

## 1. Get the code

The scripts live in `tools/implant_loosening/` of the project repo.

In [ ]:
!git clone https://github.com/Waiwaitiyyt/BMEG5551_2.git /content/project
%cd /content/project/tools/implant_loosening
!ls

## 2. Get the data

Option A downloads straight from Kaggle (upload your `kaggle.json` API token when asked).
Option B is for a copy you already put in Google Drive.

In [ ]:
# Option A - Kaggle
from google.colab import files
import os, pathlib

if not pathlib.Path("/root/.kaggle/kaggle.json").exists():
    print("Upload kaggle.json (Kaggle -> Settings -> API -> Create New Token)")
    files.upload()
    os.makedirs("/root/.kaggle", exist_ok=True)
    os.replace("kaggle.json", "/root/.kaggle/kaggle.json")
    os.chmod("/root/.kaggle/kaggle.json", 0o600)

!pip -q install kagglehub
import kagglehub
DATA_ROOT = kagglehub.dataset_download("tawsifurrahman/aseptic-loose-hip-implant-xray-database")
print("dataset at:", DATA_ROOT)

In [ ]:
# Option B - Google Drive (skip if you used Option A)
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_ROOT = '/content/drive/MyDrive/implant_xray'   # folder holding Control/ and Loose/

In [ ]:
# Sanity check: the loader finds the class folders wherever they are nested
from data import find_data_root, list_samples, CLASS_NAMES
from pathlib import Path

root = find_data_root(Path(DATA_ROOT))
samples = list_samples(root)
print(root)
print({name: sum(1 for s in samples if s.label == i) for i, name in enumerate(CLASS_NAMES)})

## 3. Look at a few scans

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import random

random.seed(0)
picks = random.sample([s for s in samples if s.label == 0], 4) + random.sample([s for s in samples if s.label == 1], 4)
fig, axes = plt.subplots(2, 4, figsize=(13, 7))
for ax, s in zip(axes.ravel(), picks):
    ax.imshow(Image.open(s.path).convert("L"), cmap="gray")
    ax.set_title(CLASS_NAMES[s.label]); ax.axis("off")
plt.tight_layout(); plt.show()

## 4. Train

5-fold cross-validation on 80% of the data, with the remaining 20% held out as a
test set that no fold ever sees. ~15 min on a T4 for the defaults.

Useful switches: `--arch efficientnet_b0`, `--img-size 384`, `--epochs 60`,
`--folds 0` (single split, much faster but noisier).

In [ ]:
!python train.py \
    --data-root "$DATA_ROOT" \
    --out-dir runs/resnet50 \
    --arch resnet50 \
    --img-size 320 \
    --epochs 40 \
    --batch-size 16 \
    --folds 5

## 5. Read the results

In [ ]:
import json
summary = json.load(open("runs/resnet50/summary.json"))
print("per-fold validation (mean +- std):")
for key, value in summary["val_mean"].items():
    print(f"  {key:<18} {value['mean']:.3f} +- {value['std']:.3f}")
print("\nheld-out test, 5-model ensemble:")
for key, value in summary["test"]["ensemble"].items():
    print(f"  {key:<18} {value}")

In [ ]:
from IPython.display import Image as ShowImage, display
for fold in range(1, 6):
    display(ShowImage(f"runs/resnet50/fold{fold}/curves.png"))

## 6. Predict on a new X-ray, with a Grad-CAM overlay

The heat map shows which region drove the decision — worth checking that it sits
on the implant/bone interface rather than on an image border or a text marker.

In [ ]:
test_image = samples[-1].path  # replace with your own file
!python predict.py --checkpoint runs/resnet50 --image "{test_image}" --cam cam_out

from IPython.display import Image as ShowImage, display
display(ShowImage(f"cam_out/{test_image.stem}_cam.png"))

## 7. Save the weights back to Drive

In [ ]:
# from google.colab import drive; drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/implant_runs && cp -r runs/resnet50 /content/drive/MyDrive/implant_runs/